### Import Libraries

In [15]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torchvision.models import resnet18

from torch.utils.data import DataLoader

### Device

In [6]:
device='cuda' if torch.cuda.is_available() else 'cpu'
print(device)

cuda


### Transforms

In [9]:
train_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225])
])

test_transform=transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485,0.456,0.406],
        std=[0.229,0.224,0.225])
])

### Dataset

In [11]:
train_dataset= datasets.CIFAR10(root='./data',train=True,download=True,transform=train_transform)
test_dataset= datasets.CIFAR10(root='./data',train=False,download=True,transform=test_transform)

100%|██████████| 170M/170M [21:24<00:00, 133kB/s]


### DataLoader

In [12]:
train_loader= DataLoader(train_dataset,batch_size=64,shuffle=True)
test_loader= DataLoader(test_dataset,batch_size=64,shuffle=False)

### Load Pretrained Model

In [16]:
model=resnet18(weights='DEFAULT')

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 182MB/s]


### Freeze All Layers

In [17]:
for param in model.parameters():
  param.requires_grad=False

### Replace Final Layer

In [18]:
model.fc=nn.Linear(
    in_features=model.fc.in_features,
    out_features=10
)

In [20]:
model=model.to(device)

### Loss Function

In [21]:
criterion=nn.CrossEntropyLoss()

### Optimizer

In [22]:
optimizer=optim.Adam(model.fc.parameters(),lr=0.001)

### Training

In [23]:
epochs=5
for epoch in range(epochs):
  model.train()
  running_loss=0

  for images,labels in train_loader:
    images=images.to(device)
    labels=labels.to(device)

    optimizer.zero_grad()
    output=model(images)

    loss=criterion(output,labels)
    loss.backward()
    optimizer.step()

    running_loss+=loss.item()

  print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss/len(train_loader)}")


Epoch 1/5, Loss: 0.8194592770789285
Epoch 2/5, Loss: 0.6161241458581231
Epoch 3/5, Loss: 0.5938914696045239
Epoch 4/5, Loss: 0.5795893651978744
Epoch 5/5, Loss: 0.5672146285433903


### Evaluation

In [24]:
model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

In [25]:
correct=0
total=0

for images,labels in test_loader:
  images=images.to(device)
  labels=labels.to(device)

  outputs=model(images)
  _,predictcted=torch.max(outputs.data,1)

  total+=labels.size(0)
  correct+=(predictcted==labels).sum().item()

print(f"Accuracy: {100*correct/total}%")


Accuracy: 81.0%


### Save Model

In [26]:
torch.save(model.state_dict(),'resnet18_cifar10.pth')

### Load Model

In [27]:
loaded_model = resnet18(weights='DEFAULT')
loaded_model.fc = nn.Linear(in_features=loaded_model.fc.in_features, out_features=10)
loaded_model.load_state_dict(torch.load('resnet18_cifar10.pth'))
loaded_model = loaded_model.to(device)
loaded_model.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  